#### CRISP-DM
- Business Understanding
- Data Understanding : Exploratory Data Analysis 
- Data Preparation : Data cleaning, normalization, Feature Engineering etc
- Model Selection
- Model Evaluation
- Model Deployment
- Monitoring 

In [1]:
#Import dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
#Data Loading and preview 
df = pd.read_csv("financial_data_1000_records.csv")
#pd.set_option("display.max_columns", 8)
df.head(5)

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,is_international
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,False
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,False
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,False
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,False
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,False


In [4]:
#Data Insoection
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         1000 non-null   int64  
 1   full_name           1000 non-null   object 
 2   age                 1000 non-null   int64  
 3   country             1000 non-null   object 
 4   account_open_date   1000 non-null   object 
 5   transaction_date    1000 non-null   object 
 6   transaction_type    1000 non-null   object 
 7   transaction_amount  1000 non-null   float64
 8   account_balance     1000 non-null   float64
 9   is_international    1000 non-null   bool   
dtypes: bool(1), float64(2), int64(2), object(5)
memory usage: 71.4+ KB


In [5]:
df.shape

(1000, 10)

In [6]:
df.describe()

,customer_id,age,transaction_amount,account_balance
count,1000.000000,1000.000000,1000.00000,1000.000000
mean,538112.094000,46.338000,2333.87194,12268.217630
std,262574.257707,16.586406,1560.76329,6004.114838
min,100404.000000,18.000000,-497.56000,112.690000
25%,301536.000000,32.000000,1010.01000,7144.540000
50%,538556.500000,46.000000,2407.58000,12098.360000
75%,762517.000000,61.000000,3679.57000,17119.870000
max,999684.000000,74.000000,4998.43000,24950.180000


In [7]:
#check null values for each feature
df.isnull().sum()

customer_id           0
full_name             0
age                   0
country               0
account_open_date     0
transaction_date      0
transaction_type      0
transaction_amount    0
account_balance       0
is_international      0
dtype: int64

In [ ]:
from ydata_profiling import ProfileReport 
profile = ProfileReport(df, title = "df prfoile report")

profile.to_notebook_iframe()

In [10]:
from ydata_profiling import ProfileReport

data_profile = ProfileReport(df, title = 'df_profile_report')

data_profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 33.60it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
df['age'].isnull().sum()

0

In [18]:
df.head(5)

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,is_international
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,False
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,False
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,False
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,False
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,False


In [13]:
df.shape

(1000, 10)

In [17]:
df['is_international'].unique()

array([False,  True])

### Task
- Rename the target column to target
- Change the values to 1 for True and 0 for false
- change the datatype to integer


In [ ]:
import pandas as pd
from typing import Union

def normalize_target_pandas(
    df: pd.DataFrame,
    target_col: str,
    inplace: bool = False
) -> pd.DataFrame:
    """
    Rename the target column to 'target', convert True/False to 1/0,
    and cast the column to integer.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    target_col : str
        Name of the existing target column.
    inplace : bool, optional (default=False)
        If True, modify the dataframe in place and return it.
        If False, work on a copy and return the new dataframe.

    Returns
    -------
    pd.DataFrame
        DataFrame with a 'target' column of integer dtype (0/1).
    """
    # --- Basic validation ---
    if target_col not in df.columns:
        raise KeyError(f"Column '{target_col}' not found in dataframe.")

    # Work on a copy unless user explicitly wants inplace modification
    if not inplace:
        df = df.copy()

    # 1) Rename the column to 'target' (if it is not already)
    if target_col != "target":
        df.rename(columns={target_col: "target"}, inplace=True)
        target_col = "target"  # update local variable

    # 2) Map values to 1/0
    # This handles bools, ints 0/1, and strings "true"/"false"/"1"/"0".
    true_values = {True, "True", "true", "TRUE", 1, "1"}
    false_values = {False, "False", "false", "FALSE", 0, "0"}

    def to_int(val):
        # Missing values stay missing
        if pd.isna(val):
            return pd.NA
        if val in true_values:
            return 1
        if val in false_values:
            return 0
        # If something unexpected is in the column, fail loudly
        raise ValueError(f"Unexpected value in target column: {val!r}")

    df["target"] = df["target"].map(to_int)

    # 3) Cast to integer type
    # Use 'Int64' (nullable integer) so it can still hold NA if needed
    df["target"] = df["target"].astype("Int64")

    return df


In [23]:
import pandas as pd
from typing import Union

import pandas as pd

def change_data(
    df: pd.DataFrame,
    target_col: str,
    inplace: bool = False
) -> pd.DataFrame:
    """
    Rename the target column to 'target', convert True/False to 1/0,
    and cast the column to integer.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    target_col : str
        Name of the existing target column.
    inplace : bool, optional (default=False)
        If True, modify the dataframe in place and return it.
        If False, work on a copy and return the new dataframe.

    Returns
    -------
    pd.DataFrame
        DataFrame with a 'target' column of integer dtype (0/1).
    """
    # Basic validation
    if target_col not in df.columns:
        raise KeyError(f"Column '{target_col}' not found in dataframe.")

    # Decide whether to work inplace or on a copy
    if inplace:
        df1 = df
    else:
        df1 = df.copy()

    # Rename now
    if target_col != "target":
        df1.rename(columns={target_col: "target"}, inplace=True)
        target_col = "target"  # update local variable

    # 2) Map values to 1/0
    # This handles bools, ints 0/1, and strings "true"/"false"/"1"/"0".
    true_values = {True, "True", "true", "TRUE", 1, "1"}
    false_values = {False, "False", "false", "FALSE", 0, "0"}

    def to_int(val):
        # retain missing values
        if pd.isna(val):
            return pd.NA
        if val in true_values:
            return 1
        if val in false_values:
            return 0
        # unexpected value
        raise ValueError(f"Unexpected value found in target column: {val!r}")

    # apply to_int function
    df1["target"] = df1["target"].map(to_int)

    # 3) Cast to integer type
    # Use 'Int64' (nullable integer) so it can still hold NA if needed
    df1["target"] = df1["target"].astype("Int64")

    return df1


    
    

In [24]:
df2 = change_data(df, target_col = "is_international", inplace = True)

In [25]:
df2.head(5)

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,target
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,0
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,0
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,0
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,0
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,0


In [26]:
df2.head(5)

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,target
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,0
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,0
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,0
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,0
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,0


In [28]:
df2_train = df2.drop(columns =['target'])
df2_train

,customer_id,full_name,age,country,...,transaction_date,transaction_type,transaction_amount,account_balance
0,221958,Jordan Smith,34,India,...,2024-03-11,ATM,1584.01,17857.01
1,771155,Casey Williams,22,France,...,2024-04-27,Online,-463.09,13702.98
2,231932,Morgan Brown,46,Germany,...,2023-01-25,POS,1622.31,20970.86
3,465838,Jordan Anderson,21,India,...,2024-10-14,ATM,3854.46,20404.84
4,359178,Jordan Thomas,27,Italy,...,2023-04-27,Online,2777.94,20849.51
...,...,...,...,...,...,...,...,...,...
995,350875,Taylor Brown,41,Canada,...,2024-07-12,Deposit,3549.26,10224.05
996,563389,Drew Thomas,32,France,...,2023-05-28,POS,1900.31,13725.68
997,170390,Riley Thomas,55,India,...,2024-10-30,ATM,3167.25,12307.39
998,807689,Alex Smith,25,India,...,2023-07-21,Online,18.80,9104.96


In [33]:
# create a metdata
from typing import List, Tuple
import pandas as pd


def get_metadata(df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Split DataFrame columns into categorical and numerical based on dtype.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame whose columns will be classified.

    Returns
    -------
    Tuple[List[str], List[str]]
        A tuple (categorical_cols, numerical_cols), where:
        - categorical_cols: columns of type object, bool, or category
        - numerical_cols : columns of numeric types (int, float, etc.)
    """
    categorical_cols = df.select_dtypes(include=["object", "bool", "category"]).columns.tolist()
    numerical_cols = df.select_dtypes(include=["number"]).columns.tolist()

    return categorical_cols, numerical_cols


            

In [38]:
column_groups = get_metadata(df2_train)

In [45]:
col = column_groups[1]
col_num = col[1:]
col_num

['age', 'transaction_amount', 'account_balance']

In [ ]:
#One hot encoding

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# X is a 2D array: rows = samples, cols = features

# Normalization to [0, 1]
mm_scaler = MinMaxScaler()
X_norm = mm_scaler.fit_transform(X)

# Standardization (mean=0, std=1)
std_scaler = StandardScaler()
X_std = std_scaler.fit_transform(X)


In [46]:
col_num = col[1:]
col_num

['age', 'transaction_amount', 'account_balance']

In [47]:
num_col = df2_train[col_num]
num_col

,age,transaction_amount,account_balance
0,34,1584.01,17857.01
1,22,-463.09,13702.98
2,46,1622.31,20970.86
3,21,3854.46,20404.84
4,27,2777.94,20849.51
...,...,...,...
995,41,3549.26,10224.05
996,32,1900.31,13725.68
997,55,3167.25,12307.39
998,25,18.80,9104.96


In [48]:
num_col

from sklearn.preprocessing import StandardScaler, MinMaxScaler

Stand_scale = StandardScaler()
X_num_Scaled = Stand_scale.fit_transform(num_col)

X_num_Scaled

array([[-0.74423438, -0.48068606,  0.93129279],
       [-1.46808043, -1.79294419,  0.23908275],
       [-0.02038833, -0.4561345 ,  1.45017162],
       ...,
       [ 0.52249621,  0.53422263,  0.00652752],
       [-1.28711892, -1.48403692, -0.52711191],
       [-1.46808043,  0.82062323,  1.68146709]])

In [49]:
num_col

from sklearn.preprocessing import MinMaxScaler 

min_max_scaler = MinMaxScaler()
min_X_Scaled = min_max_scaler.fit_transform(num_col)
min_X_Scaled

array([[0.28571429, 0.37874341, 0.71441679],
       [0.07142857, 0.00627185, 0.54716841],
       [0.5       , 0.38571213, 0.83978574],
       ...,
       [0.66071429, 0.66681526, 0.49097956],
       [0.125     , 0.09395214, 0.36204423],
       [0.07142857, 0.74810726, 0.89567021]])

## FEATURE ENGINEERING

In [51]:
df2
df2

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,target
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,0
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,0
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,0
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,0
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,0
...,...,...,...,...,...,...,...,...,...
995,350875,Taylor Brown,41,Canada,...,Deposit,3549.26,10224.05,0
996,563389,Drew Thomas,32,France,...,POS,1900.31,13725.68,0
997,170390,Riley Thomas,55,India,...,ATM,3167.25,12307.39,0
998,807689,Alex Smith,25,India,...,Online,18.80,9104.96,0


In [53]:
df_prep = df2.copy()
df_prep.head(5)

,customer_id,full_name,age,country,...,transaction_type,transaction_amount,account_balance,target
0,221958,Jordan Smith,34,India,...,ATM,1584.01,17857.01,0
1,771155,Casey Williams,22,France,...,Online,-463.09,13702.98,0
2,231932,Morgan Brown,46,Germany,...,POS,1622.31,20970.86,0
3,465838,Jordan Anderson,21,India,...,ATM,3854.46,20404.84,0
4,359178,Jordan Thomas,27,Italy,...,Online,2777.94,20849.51,0


In [54]:
X = df_prep.drop(columns = ['target'])
y = df_prep['target']

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X,y,stratify = y, random_state = 42, test_size = 0.20)

In [57]:
X_train.columns

Index(['customer_id', 'full_name', 'age', 'country', 'account_open_date',
       'transaction_date', 'transaction_type', 'transaction_amount',
       'account_balance'],
      dtype='object')

In [58]:
X_p = X_train.copy()

In [59]:
#categorcial columns
X_p_cat = X_p.select_dtypes(exclude = 'number').columns
X_p_cat 


Index(['full_name', 'country', 'account_open_date', 'transaction_date',
       'transaction_type'],
      dtype='object')

In [61]:
import pandas as pd

cat_col = ['full_name', 'country', 'account_open_date', 'transaction_date',
       'transaction_type']

X_p_enc = pd.get_dummies( X_p, columns = cat_col, drop_first = True)
X_p_enc

,customer_id,age,transaction_amount,account_balance,...,transaction_type_Deposit,transaction_type_Online,transaction_type_POS,transaction_type_Transfer
805,663586,31,3869.67,14388.59,...,False,False,False,False
498,327897,18,-250.28,12945.01,...,False,False,False,False
914,251456,54,4611.09,11140.77,...,False,True,False,False
437,788105,72,4751.66,8916.17,...,False,False,False,True
449,182074,70,1624.78,20194.01,...,True,False,False,False
...,...,...,...,...,...,...,...,...,...
951,238311,40,2798.74,12174.45,...,False,False,True,False
650,712956,21,-92.77,12839.97,...,False,False,True,False
735,450605,26,3827.23,6469.95,...,False,False,True,False
36,970910,42,333.71,3188.58,...,False,False,False,True


In [62]:
X_p_enc_bool_col = X_p_enc.select_dtypes(include = 'bool').columns
X_p_enc[X_p_enc_bool_col] = X_p_enc[X_p_enc_bool_col].astype(int)


In [63]:
X_p_enc

,customer_id,age,transaction_amount,account_balance,...,transaction_type_Deposit,transaction_type_Online,transaction_type_POS,transaction_type_Transfer
805,663586,31,3869.67,14388.59,...,0,0,0,0
498,327897,18,-250.28,12945.01,...,0,0,0,0
914,251456,54,4611.09,11140.77,...,0,1,0,0
437,788105,72,4751.66,8916.17,...,0,0,0,1
449,182074,70,1624.78,20194.01,...,1,0,0,0
...,...,...,...,...,...,...,...,...,...
951,238311,40,2798.74,12174.45,...,0,0,1,0
650,712956,21,-92.77,12839.97,...,0,0,1,0
735,450605,26,3827.23,6469.95,...,0,0,1,0
36,970910,42,333.71,3188.58,...,0,0,0,1


In [ ]:
from sklearn

In [ ]:
### Change the dataype of an existing column in the dataframe

## MODEL SECLTION AND EVALUATION

In [50]:
from sklearn.model_selection import train_test_split

